# Lab 10 — Explainable Cyber ML with Permutation Importance

*AI Cybersecurity Masterclass · Lab 10 · AI60secLab*

**Research question:** Which original telemetry fields does the fitted detector rely on most for held-out performance?

Permutation importance measures how much performance changes after a feature is shuffled.

It measures model reliance under the tested distribution—not causation.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import matplotlib.pyplot as plt

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.explainability import permutation_importance_frame

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

model = random_forest_pipeline(X_train)
model.fit(X_train, y_train)

In [ ]:
f1_importance = permutation_importance_frame(
    model,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=5,
    random_state=42,
    max_samples=0.25,
)

display(f1_importance.head(20).round(5))

In [ ]:
top = f1_importance.head(15).sort_values("importance_mean")

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(
    top["feature"],
    top["importance_mean"],
    xerr=top["importance_std"],
)
ax.set_xlabel("F1 decrease after shuffling")
ax.set_title("Held-out permutation importance")
plt.tight_layout()
plt.show()

In [ ]:
recall_importance = permutation_importance_frame(
    model,
    X_test,
    y_test,
    scoring="recall",
    n_repeats=5,
    random_state=42,
    max_samples=0.25,
)

comparison = (
    f1_importance[["feature", "importance_mean"]]
    .rename(columns={"importance_mean": "f1_importance"})
    .merge(
        recall_importance[["feature", "importance_mean"]]
        .rename(columns={"importance_mean": "recall_importance"}),
        on="feature",
        how="outer",
    )
    .fillna(0)
)

display(
    comparison
    .sort_values("recall_importance", ascending=False)
    .head(20)
    .round(5)
)

## Security interpretation

For the top features, ask:
- what telemetry does the feature represent?
- could an attacker influence it?
- would it remain available in production?
- could correlated features share the same signal?
- does importance change when scoring recall instead of F1?